In [ ]:
import pandas as pd

In [ ]:
#Colunas originais do dataset

email_col= "Email"
telefone_col= " TelefoneAjustado"
df_rd 
   email_limpo


In [ ]:
arquivo = "../data/raw/BASE COMPLETA - 01.2024-09.2025.csv"
df=pd.read_csv(
    arquivo,
    sep=";",
    encoding="latin1",
)
df.head()

In [ ]:
import re #Biblioteca padrão de expressões regulares (regex)
import pandas as pd
import unicodedata #Biblioteca padrão do Python para lidar com unicode(acentos,caracteres especiais) 
 

In [ ]:
#Função para limpar nome de coluna
# re.sub (padrão, substituto, texto) "Procure um padrão dentro do texto e substitua por outra coisa"
def limpar_nome_coluna(col:str) -> str:  #Diz que a coluna é texto e que a função devolve texto
    col=str(col).strip() #Reforço que a coluna é string e uso .strip para remover espaços
    col=re.sub(r"(?<=[a-z0-9])(?=[A-Z])"," ", col) #Este é um padrão mais avançado, ele insere um espaço entre letra minúscula e número e a outra entre letra maiúscula e número. Sintaxe, r significa: raw string
    # diz, náo interprete \ como espaço " ". Lookbehind (?<=...) "verifique o que vem antes mas não consuma, A a Z e 0 a 9. Lookhead "verifique o que vem depois mas nõs consuma"[A-Z] qualquer letra maiúscula 
    # existe qualquer letra maiúscula depois?. Até esse ponto somente identificação por fim após o parenteses o re.sub fala para substituir a vírgula por espaço
    col=re.sub(r"[_\-\/]+", " ", col)  #Pega _ ou - ou/ e troca por espaço, é usada a barra invertida para remover a ambiguidade e demonstrar que aquele é só o sómbolo mesmo.
    col=re.sub(r"\s+", " ", col). strip() # \s+ significa "um ou mais espaços ou tabs" troca tudo por um espaço só 

In [ ]:
## 3. Padronização de texto interno (strings)
#Objetivo: Remover espaços invisíveis, múltiplos espaços e ruídos de formação sem alterar os significado dos dados
def limpar_texto(x): #X é um placeholder, indica um valor qualquer vindo de uma célula do dataframe, ele precisa existir para ser o agente da iteração, a limpeza de texto passará por todos os valores do dataframe
    if pd.isna(x): # isna mean ïs not available" se o valor não estiver disponível ele retorna Nan ou None e não limpa pois não tem valor ali 
        return x
    s= str(x) # Indicamos que a partir de agora vamos trabalhar com texto
    s=s.replace("\u00a0", " ") # Este é um caractere unicode chamado de Non-breaking space, seria trocar o espaço da máquina por um espaço normal
    s=re.sub(r"\s+"," ",s) # Isso significa qualquer sequencia de espaços várias vezes
    return s.strip() # Retorna o s limpo e o último strip garante a limpeza das bordas

In [ ]:
# Nesta parte vamos criar uma lista com os nomes das colunas que contém a palavra "email", o df.columns é um index do pandas que se comporta como:
# uma lista de nomes de colunas 
print([c for c in df.columns if "Email" in c] or ("E-mail" in c))


In [ ]:
# Tínhamos criado uma lista contendo as colunas de email, decidi trabalhar com a "Email" como coluna principal
email_col= "Email"
email_col 

In [ ]:
df[email_col]=(       #Normalização de valores, estou pegando uma série do pandas que representa uma coluna inteira, o astype converte a coluna para str
   df[email_col]     # O str. garante que as ações serão aplicadas elemento por elemento. str.trip() - tira os espaços no início e no fim.  str.lower() converte tudo para min
   .astype("string")
   .str.strip()
   .str.lower()
)

In [ ]:
df[email_col].head(10)
df[email_col].isna().sum


In [ ]:
#Normalização telefone
print([c for c in df.columns if "tel" in c.lower()])

In [ ]:
telefone_col= "TelefoneAjustado"
telefone_col

In [ ]:
#Normalização de telefone
def normalizar_telefone(x):
    if pd.isna(x):
        return(x)
    
    s=str(x)
    #remove o que não é díget_ipython
    s=re.sub(r"\D", " ",s)
    return s 

df[[telefone_col]].head(10)

In [ ]:
df[telefone_col] = df[telefone_col].apply(normalizar_telefone)
df[telefone_col].head(10)
df[telefone_col].isna().sum()


In [ ]:
df.value_counts()

In [ ]:
df[telefone_col].value_counts()

In [ ]:
df[email_col].value_counts()

In [ ]:
total_registros= len(df) #Primeiro definimos o tamanho da lista de total de registros
email_preenchidos = df[email_col].notna().sum() # Aqui contamos com o "notna" - "notnan" ou "notnulo" ele me da um valor booleano em que os não nulos são True e os Nulos são False
email_vazios= df[email_col].isna().sum() #Por fim contamos os que são nulos e comparamos

In [ ]:
total_telefone= len(df) #Definição de quantidade de números de telefone
telefone_preenchidos= df[telefone_col].notna().sum() #Quantos espaços estão preenchidos
telefone_vazios = df[telefone_col].isna().sum() #Quantos espaços são nulos



In [ ]:
ambos = df[df[email_col].notna() & df[telefone_col].notna()] # Colocar o df dentro de um parenteses maior garante que o data frame geral filtre tudo que é definifo dentro do booleano seguinte
somente_telefone= df[df[email_col].isna() & df[telefone_col].notna()] # O caractere $ é um elemento de lógica booleana vetorizada, avalia os dados linha a linha e guarda e soma 
# os resultados, apenas se a condição estabelecida for verdadeira
somente_email= df[df[email_col].notna() & df[telefone_col].isna()]

print(len(ambos))
print(len(somente_telefone))
print(len(somente_email))

In [ ]:
print(df[telefone_col].nunique()) #Aqui com a função nunique nós pedimos os valores únicos de cada telefone 
print(telefone_preenchidos)


In [ ]:
base_pessoas=( # Criei uma variável de base de pessoas e agrupei nela os telefones, em size() em groupby ele conta "para cada grupo de telefone quantas linhas existe
    df
    .groupby(telefone_col)
    .size()
    .rename("n_interacoes") 
    .reset_index()   #Todo esse processo gera uma series e não uma coluna então o reset index garante tirar o índice e criar uma coluna
)

print(base_pessoas.head())
print(base_pessoas["n_interacoes"].describe().round(2))

# O describe mostra que a média de aparições é de baixa duplicação, mostra que 75% das pessoas tem somente uma aparição o que é bem consistente, temos outliers de - std - 
# desvio padrão e no número máximo, o desvio pardão explode por ter uma ocorrência muito alta se comparada com as outras, manteremos como um dado a ser aprofundado.

In [ ]:
# Buckets ou faixa de recorrência - Scoring - Quem são as pessoas em termos de frequência?

bins=[0,1,3,10,50, float("inf")] #O bins define o intervalo numérico e o labels define as etiquetas. O "inf" é uma garantia que nenhum valor fique de fora nem os extremos, vem de infinite
labels=[
    "1_interacao",
    "2_3_interacoes",
    "4_10_interacoes",
    "11_50_interacoes",
    "51+_interacoes"
]

base_pessoas["faixa_recorrencia"]=pd.cut( # O pd.cut é responsável por cortar a coluna contínua dentro do dataframe usando os intervalos definidos em bins e labels 
    # bins define onde cortar e labels como chamar cada pedaço. 
    base_pessoas["n_interacoes"],
    bins=bins,
    labels=labels
)



In [ ]:
##Limpeza e adequação - Email
df_rd=df.copy()

df_rd["email_limpo"]=(
    df_rd[email_col]
    .astype("string")
    .str.strip()
    .str.lower()
)

df_rd.loc[df_rd["email_limpo"] == "", "email_limpo"] = pd.NA
df_rd[[email_col, "email_limpo"]].head(10)

#Aqui criei um novo dataframe cópia do anterior, para fazer a limpeza em segurança, defini que tudo será string, isso ajuda para não termos problemas com os números, removi espaços e por fim padronizei
#tudo em minúsculo. Também transformei vazio em NA.




In [ ]:
df_rd.columns
print(telefone_col)

In [ ]:
import pandas as pd
import numpy as np

tel=(
    df_rd[telefone_col]
    .astype("string")
    .str.strip()
    .str.replace(r"\D+","", regex=True)
    .replace("", pd.NA)
)




In [ ]:
def padroniza_tel_com_ddi(x):
    if pd.isna(x):
        return pd.NA
    
    if len(x) in (10,11):
        return "55" + x

    if x.startswith("55") and len(x) in (12,13):
        return x

    if 8<= len(x) <=15:
        return x

    return pd.NA

df_rd["telefone_limpo"]= tel.apply(padroniza_tel_com_ddi)

df_rd["telefone_limpo"].str[:2].value_counts(dropna=False).head(10)

In [ ]:
df_rd["chave_contato"]=df_rd["email_limpo"]
df_rd.loc[df_rd["chave_contato"].isna(), "chave_contato"] = df_rd["telefone_limpo"]

df_rd["score_completude"]= df_rd.notna().sum(axis=1)
df_rd=df_rd.sort_values(by=["chave_contato", "score_completude"], ascending=[True,False])

df_dedupe = (
    df_rd
    .dropna(subset=["chave_contato"])
    .drop_duplicates(subset=["chave_contato"], keep="first")
)

print("Antes:", len(df_rd))
print("Depois(dedupe):", len(df_dedupe))

In [ ]:
print("Linhas antes:", len(df_rd))
print("Linhas depois:", len(df_dedupe))

print("Chaves únicas:", df_dedupe["chave_contato"].nunique())

print("Sem email e sem telefone:", df_rd["chave_contato"].isna().sum())

In [ ]:
cols_keep = [
    "Hóspede",
    "email_limpo",
    "telefone_limpo",
    "CidadeUF",
    "OrigemReserva",
    "TipoUH",
    "GrupoHospedagem",
    "IdIntegração",
    "UHC",
]

cols_keep=[c for c in cols_keep if c in df_dedupe.columns]
df_export=df_dedupe[cols_keep].copy()

rename_map={
    "Hóspede":"nome",
    "CidadeUF":"cidade_uf",
    "OrigemReserva":"origem_reserva",
    "MeioComunição": "meio_comunicacao",
    "SegmentoMercado": "segmento_mercado",
    "TipoUH":"tipo_uh",
    "GrupoHospedagem":"grupo_hospedagem",
    "IdIntegração":"id_integracao"
}
df_export=df_export.rename(columns=rename_map)

In [ ]:
print(df_export.isna().mean().sort_values(ascending=False).head(10))
print(df_export["telefone_limpo"].astype("string").str[:2].value_counts(dropna=False).head(15))

In [ ]:
df_export.to_csv("rd_base_limpa_dedupe.csv", index=False, encoding="utf-8-sig")
print("Gerado:rd_base_limpa_dedupe.csv")
